In [ ]:
import os
import io
import boto3
import joblib
from dotenv import load_dotenv
from botocore.exceptions import ClientError

import numpy as np
import pandas as pd

from category_encoders import TargetEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.metrics import precision_recall_curve, auc

from imblearn.over_sampling import SMOTE

import xgboost as xgb

In [ ]:
df = pd.read_csv("../dataframes/raw_data/loan_approval_data.csv")

In [ ]:
df.head()

In [ ]:
df.drop(["Applicant_ID", "Employer_Category"], inplace=True, axis=1)

In [ ]:
df.columns

In [ ]:
cols_for_ohe = ["Marital_Status", "Education_Level", "Gender", "Loan_Purpose"]

ordinal_cols = ["Property_Area"]
ordinal_cat = ["Rural", "Semiurban", "Urban"]

target_cols = ["Employment_Status"]

In [ ]:
class DataPreprocessor:
    def __init__(
        self,
        cols_for_ohe: list[str],
        ordinal_cols: list[str],
        ordinal_cat: list[str],
        target_cols: list[str],
    ):
        self.cols_for_ohe = cols_for_ohe
        self.ordinal_cols = ordinal_cols
        self.ordinal_cat = ordinal_cat
        self.target_cols = target_cols

    def feature_extraction(self, data: pd.DataFrame):
        data["total_income"] = data["Applicant_Income"] + data["Coapplicant_Income"]
        data["approximate_loan_amount"] = data["Loan_Amount"] / (
            data["Existing_Loans"] + 1
        )
        data["payment_capacity"] = ((data["total_income"]) + data["Savings"]) / (
            data["Loan_Amount"]
            * (data["Existing_Loans"] + 1)
            / (data["Credit_Score"] + 1)
        )
        data["collateral_ratio"] = data["Collateral_Value"] / (data["Loan_Amount"] + 1)
        data["income_to_loan"] = (data["total_income"]) / (data["Loan_Amount"] + 1)
        data["risk_score"] = (data["DTI_Ratio"] * data["Loan_Amount"]) / (
            data["Credit_Score"] + 1
        )
        return data

    def fit(self, X_train: pd.DataFrame, y_train: pd.Series):
        X_train = self.feature_extraction(X_train)

        self.ohe_scaler = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
        self.ohe_scaler.fit(X_train[self.cols_for_ohe])
        ohe_cols = self.ohe_scaler.get_feature_names_out(self.cols_for_ohe)
        X_ohe = pd.DataFrame(
            self.ohe_scaler.transform(X_train[self.cols_for_ohe]),
            columns=ohe_cols,
            index=X_train.index,
        )
        X_train = pd.concat([X_train.drop(columns=self.cols_for_ohe), X_ohe], axis=1)

        self.oe_scaler = OrdinalEncoder(categories=[self.ordinal_cat])
        self.oe_scaler.fit(X_train[self.ordinal_cols])
        X_train[self.ordinal_cols] = self.oe_scaler.transform(
            X_train[self.ordinal_cols]
        )

        self.target_enc = TargetEncoder(smoothing=5)
        self.target_enc.fit(X_train[self.target_cols], y_train)
        X_train[self.target_cols] = self.target_enc.transform(X_train[self.target_cols])

        return X_train, y_train

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        X = self.feature_extraction(X)

        encoded_ohe = self.ohe_scaler.transform(X[self.cols_for_ohe])
        ohe_cols = self.ohe_scaler.get_feature_names_out(self.cols_for_ohe)
        X = pd.concat(
            [
                X.drop(columns=self.cols_for_ohe),
                pd.DataFrame(encoded_ohe, columns=ohe_cols, index=X.index),
            ],
            axis=1,
        )

        X[self.ordinal_cols] = self.oe_scaler.transform(X[self.ordinal_cols])

        X[self.target_cols] = self.target_enc.transform(X[self.target_cols])
        return X

    def resample(self, X: pd.DataFrame, y: pd.Series):
        smote = SMOTE(random_state=42)
        X_res, y_res = smote.fit_resample(X, y)
        return X_res, y_res

In [ ]:
X = df.drop(columns=["Loan_Approved"])
y = df["Loan_Approved"].map({"No": 0, "Yes": 1})

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
)

In [ ]:
preprocessor = DataPreprocessor(
    cols_for_ohe=cols_for_ohe,
    ordinal_cols=ordinal_cols,
    ordinal_cat=ordinal_cat,
    target_cols=target_cols,
)

In [ ]:
X_train_processed, y_train_processed = preprocessor.fit(X_train, y_train)

In [ ]:
X_train_res, y_train_res = preprocessor.resample(X_train_processed, y_train_processed)

In [ ]:
print(y_train_res.dtype)
print(y_train_res.unique())

In [ ]:
print(X_train_res.dtypes)

In [ ]:
X_test.columns

In [ ]:
X_test_processed = preprocessor.transform(X_test)

In [ ]:
scale_pos_weight = len(np.where(y_train == 0)) / len(np.where(y_train == 1))

In [ ]:
model = xgb.XGBClassifier(
    device="cuda:0",
    learning_rate=0.05,
    n_estimators=800,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    reg_alpha=0.6,
    scale_pos_weight=scale_pos_weight,
    tree_method="hist",
)

In [ ]:
model.fit(X_train_res, y_train_res)
preds = model.predict(X_test_processed)

In [ ]:
metrics = {}

metrics["precision_vals"], metrics["recall_vals"], metrics["pr_thresholds"] = (
    precision_recall_curve(y_test, preds)
)
metrics["pr_auc"] = auc(metrics["recall_vals"], metrics["precision_vals"])

In [ ]:
print(metrics["pr_auc"])

In [ ]:
load_dotenv("../env")

MINIO3_PORT = os.getenv("S3_PORT_EXPOSE")
MINIO3_ACCESS_KEY_ID = os.getenv("MINIO_ROOT_USER")
MINIO3_SECRET_ACCESS_KEY = os.getenv("MINIO_ROOT_PASSWORD")

In [ ]:
s3_client = boto3.client(
    "s3",
    endpoint_url=f"http://localhost:{MINIO3_PORT}",
    aws_access_key_id=MINIO3_ACCESS_KEY_ID,
    aws_secret_access_key=MINIO3_SECRET_ACCESS_KEY,
)

In [ ]:
def upload_to_s3(s3_client, bucket_name: str, file_obj, key: str):
    try:
        s3_client.head_bucket(Bucket=bucket_name)
    except ClientError:
        s3_client.create_bucket(Bucket=bucket_name)

    s3_client.put_object(Bucket=bucket_name, Key=key, Body=file_obj)

In [ ]:
def save_object(s3_client, obj, bucket, key):
    buffer = io.BytesIO()
    joblib.dump(obj, buffer)
    buffer.seek(0)

    upload_to_s3(s3_client, bucket, buffer, key)

In [ ]:
save_object(s3_client, preprocessor, "inference-bucket", "preprocessor_1.pkl")

In [ ]:
save_object(s3_client, model, "inference-bucket", "xgb_model_1.pkl")